In [ ]:
import numpy as np
import pandas
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import *
from tensorflow.keras.layers import *
from tensorflow.keras import Sequential
from tensorflow.keras.datasets import mnist

In [ ]:
dataset = pandas.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train.shape, y_train.shape

In [ ]:
dataset

In [ ]:
y = dataset.iloc[:, 0]
X = np.array(dataset.iloc[:, 1:]).reshape((42000, 28, 28, 1))
X_train = X_train.reshape((60000, 28, 28, 1))
X.shape, X_train.shape

In [ ]:
X_train = np.concatenate([X_train, X], axis = 0)
y_train = np.concatenate([y_train, y], axis = 0)
X_train.shape

In [ ]:
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
model = Sequential([
    Dense(64, activation = 'relu', input_shape = (28, 28, 1)),
    Conv2D(32, kernel_size = (3, 3), activation = 'relu', padding = 'same'),
    MaxPooling2D(pool_size = (2, 2), strides = 2, padding = 'valid'),
    Conv2D(64, kernel_size = (5, 5), activation = 'relu', padding = 'same'),
    MaxPooling2D(pool_size = (2, 2), strides = 2, padding = 'valid'),
    Flatten(),
    Dense(32, activation = 'relu'),
    Dense(10, activation = 'softmax')
])

In [ ]:
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs = 5)

In [ ]:
prob = model.predict(X_test)
y_pred = np.array([np.argmax(it) for it in prob])
print(f1_score(y_pred, y_test, average = 'macro'))
CM = confusion_matrix(y_pred, y_test)
sns.heatmap(CM, annot = True, fmt = '.0f')
plt.show()

In [ ]:
test_ds = pandas.read_csv("/kaggle/input/digit-recognizer/test.csv")
test_ds = np.array(test_ds).reshape((-1, 28, 28, 1)) / 255
y_pred = model.predict(test_ds)
predictions = []
for it in y_pred:
    predictions.append(np.argmax(it))

In [ ]:
submission = pandas.DataFrame({
    "ImageId":range(1, 28001),
    "Label":predictions
})
submission

In [ ]:
submission.to_csv('submission.csv', index = False)